In [1]:
import sys
sys.path.append("..")

import pandas as pd

short_ic_ranking = pd.read_excel("../exports/RANKING_short_iron_condor.xlsx")

# Filter out gap-risk-flagged AND volatility-persistence-flagged stocks
clean_candidates = short_ic_ranking[
    (short_ic_ranking["gap_risk_flag"].isna() | (short_ic_ranking["gap_risk_flag"] == "")) &
    (short_ic_ranking["volatility_persistence_flag"].isna() | (short_ic_ranking["volatility_persistence_flag"] == ""))
]
top_25 = clean_candidates.head(25)

must_include = short_ic_ranking[short_ic_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[
    (must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")) &
    (must_include["volatility_persistence_flag"].isna() | (must_include["volatility_persistence_flag"] == ""))
]

final_candidates = pd.concat([top_25, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,short_ic_score,gap_risk_flag,Total flips (closed),Median monthly range_pct,recency_weighted_range_pct,Largest monthly range_pct,Median max_adverse_pct,recency_weighted_adverse_pct,Worst max_adverse_pct (4yr),Median duration (days),Worst single gap (4yr),last_month_range_pct,volatility_persistence_flag
0,NIFTY,98.2,NaN,76,5.36,5.04,12.39,1.06,0.96,5.95,15.2,2.32,3.87,NaN
1,SENSEX,97.5,NaN,76,5.50,5.50,12.32,1.08,1.38,5.55,15.6,2.31,4.17,NaN
2,FINNIFTY,97.4,NaN,80,6.15,5.48,15.57,1.28,1.17,7.04,14.2,2.35,5.48,NaN
3,MIDCPNIFTY,96.2,NaN,80,7.02,6.47,16.90,1.87,1.56,7.40,14.0,2.84,4.45,NaN
4,NIFTYNXT50,93.6,NaN,82,7.44,6.80,18.74,1.23,1.10,9.24,15.1,2.80,4.47,NaN
5,BANKNIFTY,89.8,NaN,70,6.20,5.58,17.01,1.52,1.62,7.15,19.6,3.73,4.47,NaN
6,SUNPHARMA,86.1,NaN,53,8.48,8.04,16.84,2.57,2.38,7.31,20.0,3.53,9.94,NaN
7,ULTRACEMCO,85.3,NaN,66,8.52,8.55,22.72,2.39,2.28,7.21,19.0,3.61,8.55,NaN
8,DABUR,84.4,NaN,65,8.59,8.96,21.12,2.41,2.41,7.34,19.2,2.66,9.56,NaN
9,GRASIM,84.2,NaN,66,9.02,8.43,21.67,2.51,2.88,6.95,18.6,3.44,5.97,NaN


In [2]:
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.fetch_data import fetch_historical_data, save_raw_data
from src.supertrend import apply_supertrend
from src.flip_analysis import detect_flips
from src.monthly_range import compute_monthly_range
from src.summary_stats import compute_summary_stats
from src.export_report import export_full_report
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        df = fetch_historical_data(kite, token, from_date=datetime(2022, 1, 1), to_date=datetime.now(), interval="4hour")
        save_raw_data(df, symbol)
        df = df.set_index("date")

        df_st = apply_supertrend(df)
        trades = detect_flips(df_st)
        monthly = compute_monthly_range(df)
        summary = compute_summary_stats(trades, monthly)
        export_full_report(trades, monthly, summary, symbol)

        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NIFTY_full_report.xlsx
[1/25] NIFTY — refreshed
[2/25] SENSEX — no token found, skipping
[3/25] FINNIFTY — no token found, skipping
[4/25] MIDCPNIFTY — no token found, skipping
[5/25] NIFTYNXT50 — no token found, skipping


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BANKNIFTY_full_report.xlsx
[6/25] BANKNIFTY — refreshed
Fetching 2022-01-01 to 2023-01-31...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/SUNPHARMA_full_report.xlsx
[7/25] SUNPHARMA — refreshed
Fetching 2022-01-01 to 2023-01-31...


...

/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/VOLTAS_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/VOLTAS_full_report.xlsx
[24/25] VOLTAS — refreshed
Fetching 2022-01-01 to 2023-01-31...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-07...
Saved 2266 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABB_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ABB_full_report.xlsx
[25/25] ABB — refreshed


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


In [3]:
from src.monthly_range_screener import screen_monthly_range

live_check = screen_monthly_range(symbols_to_refresh)
live_check.sort_values("current_month_range_pct", ascending=False)

,symbol,days_into_month,current_month_range_pct,median_monthly_range,75th_pct_monthly_range,90th_pct_monthly_range,already_past_median,already_past_75th,already_past_90th
24,ABB,7,8.99,13.28,18.02,24.21,False,False,False
23,VOLTAS,7,6.74,12.26,16.89,23.97,False,False,False
21,BOSCHLTD,7,6.41,10.32,12.61,19.83,False,False,False
19,GAIL,7,5.62,12.66,16.70,20.12,False,False,False
15,SHREECEM,7,5.51,9.86,13.73,17.21,False,False,False
13,JSWSTEEL,7,5.18,10.28,12.98,15.26,False,False,False
11,COLPAL,7,4.98,9.00,11.94,18.98,False,False,False
8,DABUR,7,4.92,8.59,10.75,15.08,False,False,False
22,LICHSGFIN,7,4.22,11.98,15.83,19.22,False,False,False
9,GRASIM,7,4.16,9.02,11.53,14.90,False,False,False
